In [1]:
from mysql import connector
import pandas as pd
import numpy as np

print(pd.__version__)
print(np.__version__)
print("설치 완료")

2.2.3
2.2.5
설치 완료


## DB 연결 기초 코드
    - 여기서부터 시작
    - pw는 mysql 접속하는 pw

In [2]:
import os
from mysql import connector

PASSWORD = 'jh0402'

try:
    with connector.connect(
        host = "localhost",
        user = "root",
        password = PASSWORD
    ) as database: 
        print(f"Database object: {database}")
except connector.Error as e: 
    print(e)

Database object: <mysql.connector.connection_cext.CMySQLConnection object at 0x000001C3B173DFD0>


## DB 연결 추천 방식

In [3]:
from dotenv import load_dotenv
import os
import mysql.connector
from mysql.connector import pooling

# .env 파일 로드
load_dotenv()

# 데이터베이스 연결 설정
db_config = {
    'host': os.getenv('MYSQL_HOST'),
    'port': int(os.getenv('MYSQL_PORT')),
    'user': os.getenv('MYSQL_USER'),
    'password': os.getenv('MYSQL_PASSWORD'),
    'database': os.getenv('MYSQL_DATABASE'),
    'charset': os.getenv('MYSQL_CHARSET')
}

# 커넥션 풀 설정
pool_config = {
    'pool_name': os.getenv('MYSQL_POOL_NAME'),
    'pool_size': int(os.getenv('MYSQL_POOL_SIZE')),
    'pool_reset_session': True,
    **db_config
}

# 커넥션 풀 생성
connection_pool = mysql.connector.pooling.MySQLConnectionPool(**pool_config)

# 커넥션 풀 사용 예시
def get_connection():
    return connection_pool.get_connection()

# 사용 예시
try:
    connection = get_connection()
    cursor = connection.cursor()

    sql_query = '''
        SELECT 1+1;
    '''
    
    # 쿼리 실행
    cursor.execute("SELECT 1+1;")
    results = cursor.fetchall()
    
    for row in results:
        print(row)
        
except mysql.connector.Error as err:
    print(f"Error: {err}")
finally:
    if 'cursor' in locals():
        cursor.close()
    if 'connection' in locals():
        connection.close()

(2,)


# 스키마 생성

In [4]:
import os
from mysql import connector

PASSWORD = 'jh0402'

try:
    with connector.connect(
        host = "localhost",
        user = "root",
        password = PASSWORD
    ) as database: 
        print(f"Database object: {database}")
        delete_db = 'DROP DATABASE book_ratings;'      # 스키마 삭제 명령어
        create_db = 'CREATE DATABASE book_ratings;'    # 스키마 생성 명령어
        with database.cursor() as cursor:
            # cursor.execute(delete_db)
            cursor.execute(create_db)

            # 현재 존재하는 스키마 목록 확인
            show_db = 'SHOW DATABASES;'
            cursor.execute(show_db)
            for db in cursor:
                print(db)
except connector.Error as e: 
    print(e)

Database object: <mysql.connector.connection_cext.CMySQLConnection object at 0x000001C3912DE850>
1007 (HY000): Can't create database 'book_ratings'; database exists


In [5]:
# 특정 스키마 지정 후 연결
import os
from mysql import connector

create_book_table = """
CREATE TABLE books(
    id INT NOT NULL AUTO_INCREMENT PRIMARY KEY,
    title VARCHAR(100),
    author VARCHAR(100),
    genre VARCHAR(100),
    release_year YEAR(4)
)
"""

try:
    with connector.connect(
        host = "localhost",
        user = "root",
        password = 'jh0402',
        database = 'book_ratings' # 차이점
    ) as database: 
        print(f"Database object: {database}")
        with database.cursor() as cursor:
            #cursor.execute(create_book_table) # 한 번 실행한 후에는 에러가 뜨므로 두 번째 실행할 때는 주석처리 해야함
            database.commit()

            # 테이블 확인
            describe_books = 'DESCRIBE books;'
            cursor.execute(describe_books)
            result = cursor.fetchall() # 모든 테이블 정보를 다 가지고 오는데,
            for col in result:         # pandas 데이터프레임으로 오는 것이 아니라 리스트 - 튜플 형태로 저장
                print(col)
            
except connector.Error as e: 
    print(e)

Database object: <mysql.connector.connection_cext.CMySQLConnection object at 0x000001C3B17D5F90>
('id', 'int', 'NO', 'PRI', None, 'auto_increment')
('title', 'varchar(100)', 'YES', '', None, '')
('author', 'varchar(100)', 'YES', '', None, '')
('genre', 'varchar(100)', 'YES', '', None, '')
('release_year', 'year', 'YES', '', None, '')


# 데이터 추가하기

In [6]:
# INSERT SINGLE RECORD
# insert_single_record, single_record는 문법임
insert_single_record = "INSERT INTO books (id, title, author, genre, release_year)\
    VALUES (%s, %s, %s, %s, %s)"
single_record = (
    "1", "Choose Yourself! Be Happy, Make Millions, Live the Dream", "James Altucher", "self-help", "2013"
    )

try: 
    # Connect to existing database
    with connector.connect(
        host = "localhost",
        user = "root",
        password = 'jh0402',
        database = "book_ratings"
    ) as existing_database:
        
        # Create cursor object
        with existing_database.cursor() as cursor:
            cursor.execute(insert_single_record, single_record)
            existing_database.commit()
        
except connector.Error as e: 
    print(e)

# INSERT MULTIPLE RECORDS
insert_multiple_records = "INSERT INTO books (id, title, author, genre, release_year)\
    VALUES (%s, %s, %s, %s, %s)"
# 리스트로 튜플 감싸는 형식으로 만들기
multiple_records = [
    (
        "2", 
        "Skip the Line: The 10,000 Experiments Rule and Other Surprising Advice for Reaching Your Goals",
        "James Altucher",
        "self-help",
        "2021"        
    ),
    (
        "3",
        "The Power of No: Because One Little Word Can Bring Health, Abundance, and Happiness",
        "James Altucher",
        "self-help",
        "2014"
    ),
    (
        "4",
        "The 48 Laws of Power",
        "Robert Greene",
        "self-help",
        "1998"
    ),
    (
        "5",
        "Mastery",
        "Robert Greene",
        "self-help",
        "2012"
    ),
    (
        "6",
        "The Art of Seduction",
        "Robert Greene",
        "self-help",
        "2001"
    ),
]

try: 
    # Connect to existing database
    with connector.connect(
        host = "localhost",
        user = "root",
        password = 'jh0402',
        database = "book_ratings"
    ) as existing_database:
        
        # Create cursor object
        with existing_database.cursor() as cursor:
            cursor.executemany(insert_multiple_records, multiple_records) # 이 부분만 문법이 다름
            existing_database.commit()
        
except connector.Error as e: 
    print(e)

1062 (23000): Duplicate entry '1' for key 'books.PRIMARY'
1062 (23000): Duplicate entry '2' for key 'books.PRIMARY'


# 데이터 조회

In [7]:
import os
from mysql import connector

PASSWORD = 'jh0402'

try:
    with connector.connect(
        host = "localhost",
        user = "root",
        password = PASSWORD,
        database = 'book_ratings'
    ) as database: 
        print(f"Database object: {database}")

        # 조회
        select_query = 'SELECT author, release_year FROM books;'
        with database.cursor() as cursor:
            cursor.execute(select_query)

            # 데이터 가져온 것을 출력
            df = cursor.fetchall()
            for result in df:
                print(result)
            
except connector.Error as e: 
    print(e)

Database object: <mysql.connector.connection_cext.CMySQLConnection object at 0x000001C3912DE710>
('James Altucher', 2013)
('James Altucher', 2021)
('James Altucher', 2014)
('Robert Greene', 1998)
('Robert Greene', 2012)
('Robert Greene', 2001)


In [8]:
# 미션: 어떤 형태로든 테이블을 조회해도 Table의 고유한 컬럼 & 데이터프레임으로 만들기
# 함수 사용
import os
from mysql import connector

PASSWORD = 'jh0402'

try:
    with connector.connect(
        host = "localhost",
        user = "root",
        password = PASSWORD,
        database = 'book_ratings'
    ) as database: 
        print(f"Database object: {database}")

        # 조회
        select_query = 'SELECT * FROM books;'
        with database.cursor() as cursor:
            cursor.execute(select_query)

            # 컬럼명 가져오기
            column_names = [desc[0] for desc in cursor.description]
            rows = cursor.fetchall()

            # 데이터 프레임 생성
            df = pd.DataFrame(rows, columns=column_names)
            print(df)

except connector.Error as e: 
    print(e)


Database object: <mysql.connector.connection_cext.CMySQLConnection object at 0x000001C3B17D5F90>
   id                                              title          author  \
0   1  Choose Yourself! Be Happy, Make Millions, Live...  James Altucher   
1   2  Skip the Line: The 10,000 Experiments Rule and...  James Altucher   
2   3  The Power of No: Because One Little Word Can B...  James Altucher   
3   4                               The 48 Laws of Power   Robert Greene   
4   5                                            Mastery   Robert Greene   
5   6                               The Art of Seduction   Robert Greene   

       genre  release_year  
0  self-help          2013  
1  self-help          2021  
2  self-help          2014  
3  self-help          1998  
4  self-help          2012  
5  self-help          2001  


# 코드에서 환경변수 읽어오기

In [9]:
import os
from mysql import connector

# 환경 변수에서 비밀번호를 읽어옴
PASSWORD = os.getenv('MYSQL_PASSWORD')

if PASSWORD is None:
    print("비밀번호 환경 변수가 설정되지 않았습니다.")
else:
    print(f"비밀번호가 정상적으로 설정되었습니다.")

비밀번호가 정상적으로 설정되었습니다.
